# Introdução
Esse notebook tem por objetivo carregar os dados limpos dos arquivos json profiles, offers, e transactions, fazer uma modelagem estatística e usá-la em um modelo de regressão logística, que prevê se um determinado perfil de cliente tem ou não propensão para compras, dado o conjunto de ofertas.

# 1. Importar Livrarias

In [0]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.functions import expr
from pyspark.sql.functions import when
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
)


# 2. Carregar tabelas limpas da etapa data processing

As tabelas estão salvas no diretório workspace.default.profile. A versão community do databricks não deixa gravar os dados diretamente no workspace.

In [0]:
profile = spark.table("workspace.default.profile_clean")
offers = spark.table("workspace.default.offers_clean")
transactions = spark.table("workspace.default.transaction_clean")

# 3. Filtrar a tabela transactions com offer received

In [0]:
offer_received = (
    transactions
    .filter(F.col("event") == "offer received")
    .select(
        F.col("account_id").alias("o_account_id"),
        F.col("time_since_test_start").alias("offer_start"),
        F.col("value.offer_id").alias("offer_id")
    )
)

In [0]:
# test
display(offer_received)

# 4. Enriquecer offer received com informações de oferta

In [0]:
offers_df = (
    offers
    .select(
        F.col("id").alias("offer_id"),
        "offer_type",
        "duration",
        "discount_value",
        "min_value"
    )
)

offer_received = (
    offer_received
    .join(offers_df, on="offer_id", how="left")
)

In [0]:
display(offer_received)

# 5. Cálculo do fim da validade da promoção

In [0]:
offer_received = offer_received.withColumn(
    "offer_end",
    F.col("offer_start") + F.col("duration")
)

In [0]:
display(offer_received.filter(F.col('offer_start')!=0))

# 6. Filtrar transacoes

In [0]:
transactions_only = (
    transactions
    .filter(F.col("event") == "transaction")
    .select(
        F.col("account_id").alias('t_account_id'),
        F.col("time_since_test_start").alias("trx_time"),
        F.col("value.amount").alias("amount")
    )
)

# 7.Verificar se houve compra
A condição usada para verificar se houve ou não compra é pegar as transações cujo valor do tempo trx_time sejam maiores do que o início da oferta mas sejam menores do que o tempo da oferta.
Isso significa que houve uma transação durante o início e antes do fim da oferta, e o cliente comprou(isso é apenas um critério e uma hipótes, já que minhas perguntas não foram respondidas)

In [0]:
joined = (
    offer_received.alias("o")
    .join(
        transactions_only.alias("t"),
        (
            (F.col("o.o_account_id") == F.col("t.t_account_id"))
            &
            (F.col("t.trx_time") >= F.col("o.offer_start"))
            &
            (F.col("t.trx_time") <= F.col("o.offer_end"))
        ),
        "left"
    )
)

In [0]:
joined = joined.select(F.col('offer_id')\
                      ,F.col('o_account_id').alias('account_id')\
                      ,F.col('offer_start')\
                      ,F.col('offer_type')\
                      ,F.col('duration')\
                      ,F.col('discount_value')\
                      ,F.col('min_value')\
                      ,F.col('offer_end')\
                      ,F.col('trx_time')\
                      ,F.col('amount'))

In [0]:
display(joined.orderBy('account_id'))

# 8. Criando o Target

o max( ) na cláusula when funciona como uma query agregativa do sql, retornando apenas uma linha

In [0]:
dataset = (
    joined
    .groupBy(
        "account_id",
        "offer_id",
        "offer_start",
        "offer_end",
        "offer_type",
        "duration",
        "discount_value",
        "min_value"
    )
    .agg(
        F.max(
            F.when(F.col("trx_time").isNotNull(), 1)
             .otherwise(0)
        ).alias("target")
    )
)

In [0]:
display(dataset)

# 9. Agregando informações do cliente

In [0]:
dataset = (
    dataset
    .join(
        profile,
        dataset.account_id == profile.id,
        "left"
    )
    .drop("id")
)

In [0]:
display(dataset)

## 9.1. Fazendo uma última limpeza 

In [0]:
dataset = dataset.filter(F.col("gender").isNotNull())
#dataset = dataset.filter(F.col("gender")!='O')
display(dataset)

10. Convertendo para o Pandas

In [0]:
df = dataset.toPandas()

In [0]:
df.head(10)

# 10. Preprarar para ML

In [0]:
X = df.drop(columns=["target"])
y = df["target"]

In [0]:
X.head(10)

# 11. Codificar variáveis categóricas

In [0]:
X = pd.get_dummies(
    X,
    columns=["gender", "offer_type"],
    drop_first=True
)

In [0]:
display(X)

# 12. Remover identificadores

In [0]:
X = X.drop(columns=["account_id", "offer_id"])

In [0]:
display(X)

13. Split de variáveis entre treino, teste e validação

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    random_state=42,
    test_size=0.2
)

# 13. Modelo de Regressão Logística

In [0]:
model = LogisticRegression(
    max_iter=1000
)

model.fit(X_train, y_train)

# 14. Avaliação do Modelo

In [0]:


pred = model.predict(X_test)

prob = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, pred))

print("ROC AUC:", roc_auc_score(y_test, prob))

In [0]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)

print(cm)

In [0]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

disp = ConfusionMatrixDisplay.from_estimator(
    model,
    X_test,
    y_test,
    cmap="Blues",
    values_format="d"
)

plt.title("Matriz de Confusão - Regressão Logística")
plt.show()